# **Canadian Lake-River Hydrofabric (CLRH) v1.0 Colab notebook using BasinMaker v3.1 Post-processing Tools**

*Last updated Apr 24, 2025 by B Tolson. Minor updates.*

This Colab notebook was adapted by Bryan Tolson & Ming Han for CLRH access and simplification via BasinMaker 3.1. Using this notebook gives you access to CLRH data and BasinMaker post-processing functions without locally installing any software on your computer. If you are using this notebook, you must be using a CLRH routing network from our CLRH website here: https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html.

**Before going any further, please go that website if you have not visited it before and browse around.  Furthermore, if you have not already, make sure you open up this link with CLRH data specifications (accessible also from the above website) and skim the file.**  
https://hydrology.uwaterloo.ca/CLRH/files/CLRH_v1_data_specifications.pdf.
**Only then proceed below.**

CLRH is a product produced by the collaborative efforts of the following fine group of people led by Bryan Tolson and James Craig of the University of Waterloo:
Simon Lin, Ming Han, Hongren Shen, Parisa Aberi, Joshua Wiebe, Madeline G. Tucker, Qiutong (Glen) Yu, Benjamin Searson, Savreen Babra, Stefan Kolev, and Mary Stapleton.

Using this notebook means that you should be citing the following publications:

*   BasinMaker software citation:  Han, M., H. Shen, B. A. Tolson, J. R. Craig, J. Mai, S. Lin, N. B. Basu, and F. Awol (2023). BasinMaker 3.0: a GIS toolbox for distributed watershed delineation of complex lake-river routing networks, Environmental Modelling and Software, 164. doi.org/10.1016/j.envsoft.2023.105688.  
*   CLRH data access citation: the CLRH website above.  Please email btolson@uwaterloo.ca for updated citation when you are finalizing your publication. A data paper is being produced.


***Please note this notebook produces outputs on temporary drive and you should download zip outputs created before closing the Colab website.***

The BasinMaker toolbox (Han et al., 2023) has been previously applied to produce comprehensive and seamless lake and river routing products across **North America (hereafter called the NA routing product or NALRP)** and **the Province of Ontario (hereafter called the OLRRP)**. Our **Canadian Lake and River Hydrofabric (CLRH)** is the latest such product and should absolutely be favoured over NALRP wherever both are available.  OLRRP ver2 uses Ontario specific geospatial data and should be favoured over CLRH if possible (although both are quite accurate). BasinMaker routing products/hydrofabrics provide hydrologically meaningful subbasin routing network topology, (sub)basin characteristics, and lake and river attributes. Importantly, these network attributes can be converted into Raven hydrologic modeling (https://raven.uwaterloo.ca/) required input files seamlessly with BasinMaker.

Here are three potentially useful webpages:

1. The CLRH website to access the data (where this notebook downloads from): https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html
2. BasinMaker webpage: http://hydrology.uwaterloo.ca/basinmaker
3. Our latest hydrofabric for the Province of Ontario (OLLRP ver2):https://uwaterloo-olrrp.shinyapps.io/OLRRP-V2/

# **Content of this notebook**

       1. Install BasinMaker (light) and deploy the Python environment;
       2. Bring a routing network into your Colab session:
          a) Download a gauge level Watershed routing network from CLRH website
          b) Download a regional level routing network from CLRH website
          c) Upload your CLRH-based routing network from another location
       3. Extract drainage area within above example catchment;
       4. RavenView to view geojsons of routing network;
       5. Simplify the routing network by filtering lakes;
       6. Simplify the routing network by increasing size of subbasins;
       7. Create land and lake HRUs;
       8. Produce Raven-required inputs; and
       9.  Appendix of useful Google Colab codes

       *Future content to be added here:*
       a) Add/remove Points of Interest in routing product (emulate Step 4 in OLRRPv2 Colab via above link) to enable users to remove Border;
       b) Create more complex HRUs (See Step 7 in OLRRPv2 Colab via above link)

# **Notes on each section in this notebook**

1. Each of the sections below contains two parts:

  One or more text blocks for clarifying some important notes - like this text block and the text blocks you have been reading above.
  
  One or more code blocks (code cells) to implement data access and BasinMaker functions.

In [1]:
# This is what a code cell looks like.  This is a comment line in the code.  No user inputs required.
# If you press the play button to the left, the code will be executed.

# You will also notice the old output of this code cell is also printed below before you even press the play button.

print("Nice work pressing the play button!")

# Note the button just below the code cell on the left has options for code cell output printing.

Nice work pressing the play button!


2. Read the text part, as well as code cell comments as well, as it forms the notebook instructions.

3. The work you need to do is to locate the **USER INPUTS** and understand them (already set at default working values for you to use) in the code cells below.  Clearly these USER INPUTS change for a different case study routing network. After trying all sections and functions below with default inputs, users are encouraged to go and modify some inputs (e.g., try a different gauge network) and repeat some of the sections.

4. Some code cells below when executed will generate many 'WARNING' messages and these can be safely ignored.

Time to start using the notebook!


# **1.0 Install BasinMaker (light) and deploy the Python environment**


*   Run this code cell in order use BasinMaker on Colab. No need to read the printed output beneath the code cell unless you get an error.
*   For those who are interested in BasinMaker software installation on your local machine, go to the BasinMaker installation guide ([here](https://basinmaker.readthedocs.io/en/latest/installation.html)) for details.  Install can be challenging.



In [2]:
# This code block installs BasinMaker on your Colab runtime.
# No user inputs, just click play button to the left.

# Note that everytime your runtime is terminated, the installed BasinMaker will be removed from your runtime.
# Thus, you will need to install BasinMaker everytime you log into Colab.

import sys
import warnings
#!python -m pip install pandas pytest scipy simpledbf netCDF4 joblib wget ipywidgets ipyleaflet fiona shapely pyproj rtree geopandas rasterstats
#!pip install --upgrade --no-cache-dir gdown
#working Oct31-24:
#!python -m pip install https://github.com/dustming/basinmaker/archive/master.zip
warnings.filterwarnings('ignore')

### load packages
from basinmaker import basinmaker
import pandas as pd
import os
import shutil # added by BT, Nov2024
import wget
import geopandas
import time
from branca.colormap import linear
import matplotlib.pyplot as plt ## only needed to plot figures
from ipywidgets import HTML,Layout,IntSlider, ColorPicker, jslink ## only needed to plot figures
from ipyleaflet import Map, GeoData, basemaps, LayersControl,Popup,Marker,Polygon,Choropleth,WidgetControl## only needed to plot figures

from basinmaker.postprocessing.plotleaflet import plot_routing_product_with_ipyleaflet
from basinmaker.postprocessing.downloadpd import Download_Routing_Product_For_One_Gauge
from basinmaker.postprocessing.downloadpdptspurepy import Download_Routing_Product_From_Points_Or_LatLon

In [ ]:
# ------------------------------------------------------------
# FUNCTION: Fix reciprocal upstream/downstream relationships
# ------------------------------------------------------------
import os
import geopandas as gpd

def fix_reciprocal_links(gdf, sub_col="SubId", down_col="DowSubId"):
    """
    Fix reciprocal SubId ↔ DowSubId loops by setting the downstream
    value of the second link to -1.
    """
    # Build pair set
    pairs = set(zip(gdf[sub_col], gdf[down_col]))

    # Mask of rows that are reciprocal (A→B where B→A also exists)
    mask = gdf.apply(lambda r: (r[down_col], r[sub_col]) in pairs, axis=1)

    # Print what is being fixed
    if mask.any():
        print("  Reciprocal rows found:")
        print(gdf[mask][[sub_col, down_col]])
    else:
        print("  No reciprocal rows found.")

    # Fix by breaking the backward link
    gdf.loc[mask, down_col] = -1

    return gdf


# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
base_path = r"D:\Zelalem\CLRH_Basin\CLRH_Basin\01_02\01_02"

shapefile_list = [
    "catchment_without_merging_lakes_v1-0.shp",
    "finalcat_info_riv_v1-0.shp",
    "finalcat_info_v1-0.shp",
    "outline.shp",
    "poi_v1-0.shp",
    "river_without_merging_lakes_v1-0.shp",
    "sl_connected_lake_v1-0.shp"
]


for shp_name in shapefile_list:

    print("\n--------------------------------------")
    print(f"Processing {shp_name}")
    print("--------------------------------------")

    shp_path = os.path.join(base_path, shp_name)

    if not os.path.exists(shp_path):
        print(f"  File not found: {shp_path}")
        continue

    # Load file
    gdf = gpd.read_file(shp_path)

    # If SubId/DowSubId not in file, skip
    if "SubId" not in gdf.columns or "DowSubId" not in gdf.columns:
        print("  Columns SubId or DowSubId not present. Skipping.")
        continue

    # Fix reciprocal loops
    gdf_fixed = fix_reciprocal_links(gdf)

    # Save back to the same file (overwrite)
    gdf_fixed.to_file(shp_path)
    print(f"  ✔ Saved fixed file: {shp_path}")

In [ ]:
## Authomate basinmaker for basin and lake aggregation
input_dir = '/scratch/zelalem/merged_shp'
output_dir = '/scratch/zelalem/merged_shp'
min_lakearea_km2 = 5
min_subbasin_km2 = 100

# List only directories (folders) in input_dir
watershed_regions = [name for name in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, name))]
#watershed_regions = [ '10TA001', '10TF001', '10VC002', '10VK001'] # ['Nelson'] #['Nelson', '06_04', '01_01'] #, '04H', 'Machanze']

#interested_lake_ids_dict = {
#    '02H': [],
#    '04H': [],
#    'Machanze': [],
#}

for watershed_region in watershed_regions:    
    folder_product_for_interested_gauges = os.path.join(input_dir,watershed_region,watershed_region)
    input_routing_product_folder = folder_product_for_interested_gauges
    
    folder_product_after_filter_lakes = os.path.join(output_dir,watershed_region,'filter_lakes')
    os.makedirs(folder_product_after_filter_lakes, exist_ok=True)

# uncomment the line below if interested_lake_ids_dict available from above
#    interested_lake_ids = interested_lake_ids_dict.get(network_name, [])
    interested_lake_ids = []
    
    try:
        # Step 1: Remove small lakes
        bm = basinmaker.postprocess()
        bm.Remove_Small_Lakes(
            path_output_folder=folder_product_after_filter_lakes,
            routing_product_folder=input_routing_product_folder,
            connected_lake_area_thresthold=min_lakearea_km2,
            non_connected_lake_area_thresthold=min_lakearea_km2,
            selected_lake_ids=interested_lake_ids,
            gis_platform="purepy",
        )
        
        # Step 2: Increase size of subbasins
        input_routing_product_folder = folder_product_after_filter_lakes
        folder_product_after_increase_catchment_drainage_area = os.path.join(output_dir,watershed_region,'increase_drainage_area')
        os.makedirs(folder_product_after_increase_catchment_drainage_area, exist_ok=True)
        
        bm = basinmaker.postprocess()
        bm.Decrease_River_Network_Resolution(
            path_output_folder=folder_product_after_increase_catchment_drainage_area,
            routing_product_folder=input_routing_product_folder,
            minimum_subbasin_drainage_area=min_subbasin_km2,
            gis_platform="purepy",
        )
        
        print(f'Processing done for network: {watershed_region}')
        
    except Exception as e:
        print(f"Error processing {watershed_region}: {e}")

In [ ]:
 Obtain selected Lake IDs done
Processing done for network: 10TA001
 Obtain selected Lake IDs done
Processing done for network: 10TF001
 Obtain selected Lake IDs done
Processing done for network: 10VC002
 Obtain selected Lake IDs done
Processing done for network: 10VK001

In [ ]:
## Authomate basinmaker basin and lake aggregation
input_dir = '/scratch/zelalem/merged_shp'
output_dir = '/scratch/zelalem/merged_shp'
min_lakearea_km2 = 5
min_subbasin_km2 = 100

# List only directories (folders) in input_dir
# watershed_regions = [name for name in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, name))]
watershed_regions = ['Mackenzie']  #['10TA001', '10TF001', '10VC002', '10VK001'] # ['Nelson'] #['Nelson', '06_04', '01_01'] #, '04H', 'Mackenzie']

#interested_lake_ids_dict = {
#    '02H': [],
#    '04H': [],
#    'Machanze': [],
#}

for watershed_region in watershed_regions:    
    folder_product_for_interested_gauges = os.path.join(input_dir,watershed_region,watershed_region)
    input_routing_product_folder = folder_product_for_interested_gauges
    
    folder_product_after_filter_lakes = os.path.join(output_dir,watershed_region,'filter_lakes')
    os.makedirs(folder_product_after_filter_lakes, exist_ok=True)

# uncomment the line below if interested_lake_ids_dict available from above
#    interested_lake_ids = interested_lake_ids_dict.get(network_name, [])
    interested_lake_ids = []
    
    try:
        # Step 1: Remove small lakes
        bm = basinmaker.postprocess()
        bm.Remove_Small_Lakes(
            path_output_folder=folder_product_after_filter_lakes,
            routing_product_folder=input_routing_product_folder,
            connected_lake_area_thresthold=min_lakearea_km2,
            non_connected_lake_area_thresthold=min_lakearea_km2,
            selected_lake_ids=interested_lake_ids,
            gis_platform="purepy",
        )
        
        # Step 2: Increase size of subbasins
        input_routing_product_folder = folder_product_after_filter_lakes
        folder_product_after_increase_catchment_drainage_area = os.path.join(output_dir,watershed_region,'increase_drainage_area')
        os.makedirs(folder_product_after_increase_catchment_drainage_area, exist_ok=True)
        
        bm = basinmaker.postprocess()
        bm.Decrease_River_Network_Resolution(
            path_output_folder=folder_product_after_increase_catchment_drainage_area,
            routing_product_folder=input_routing_product_folder,
            minimum_subbasin_drainage_area=min_subbasin_km2,
            gis_platform="purepy",
        )
        
        print(f'Processing done for network: {watershed_region}')
        
    except Exception as e:
        print(f"Error processing {watershed_region}: {e}")

 Obtain selected Lake IDs done


In [ ]:
"""
Final--BasinMaker-style merge utility

- Scans all */increase_drainage_area folders
- Robust CRS handling
- Fixes invalid geometries
- Harmonizes geometry types for ESRI Shapefile
- Forces Fiona I/O (avoids field-width warnings)
- Outputs to merged_shp/, consistent with BasinMaker deliverables
"""

from pathlib import Path
import geopandas as gpd
import pandas as pd
from shapely.geometry import (
    Polygon, MultiPolygon,
    LineString, MultiLineString,
    Point, MultiPoint
)

# ---------------------------------------------------------------------
# Geometry utilities (mirrors BasinMaker-style safety)
# ---------------------------------------------------------------------

def fix_invalid_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Fix invalid geometries where possible."""
    if not gdf.is_valid.all():
        gdf["geometry"] = gdf["geometry"].buffer(0)
    return gdf


def enforce_single_geom_type(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Convert mixed geometries to a consistent multi-geometry type
    so Shapefile writing never fails.
    """
    geom_types = set(gdf.geometry.geom_type)

    if geom_types <= {"Polygon", "MultiPolygon"}:
        gdf["geometry"] = gdf.geometry.apply(
            lambda g: MultiPolygon([g]) if isinstance(g, Polygon) else g
        )

    elif geom_types <= {"LineString", "MultiLineString"}:
        gdf["geometry"] = gdf.geometry.apply(
            lambda g: MultiLineString([g]) if isinstance(g, LineString) else g
        )

    elif geom_types <= {"Point", "MultiPoint"}:
        gdf["geometry"] = gdf.geometry.apply(
            lambda g: MultiPoint([g]) if isinstance(g, Point) else g
        )

    return gdf


# ---------------------------------------------------------------------
# Core merge routine
# ---------------------------------------------------------------------

def merge_all_basinmaker_layers(
    root_dir: str,
    layer_name: str,
    target_crs: str = "EPSG:4326"      #"EPSG:3979"
):
    root = Path(root_dir)

    # BasinMaker layout: */increase_drainage_area/<layer>
    files = sorted(root.glob(f"*/increase_drainage_area/{layer_name}"))
#    files = sorted(root.glob(f"*/*/{layer_name}"))    # for the raw data

    if not files:
        print(f"Not found: {layer_name}")
        return

    print(f"\nFound {len(files)} files → {layer_name}")

    gdfs = []
    for i, fp in enumerate(files, 1):
        basin_name = fp.parts[-3]
        print(f"  [{i:3d}] {basin_name:<15} reading...", end="")

        gdf = gpd.read_file(fp)

        # CRS handling (robust, BasinMaker-style)
        if gdf.crs is None:
            gdf = gdf.set_crs(target_crs)
        elif gdf.crs.to_string() != target_crs:
            gdf = gdf.to_crs(target_crs)

        gdf = fix_invalid_geometries(gdf)
        gdf = enforce_single_geom_type(gdf)

        gdfs.append(gdf)
        print(" OK")

    merged = gpd.GeoDataFrame(
        pd.concat(gdfs, ignore_index=True),
        crs=target_crs
    )

    # Output directory (BasinMaker-consistent naming)
    out_dir = root / "merged_shp"
    out_dir.mkdir(exist_ok=True)

    out_file = out_dir / layer_name

    print(f"\nSaving {len(merged):,} features → {out_file}")

    # IMPORTANT: force Fiona (avoids field-width warnings)
    merged.to_file(
        out_file,
        driver="ESRI Shapefile",
        engine="fiona"
    )

    print("SUCCESS")


# ---------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------

if __name__ == "__main__":

    ROOT = r"D:\Zelalem\CLRH_Basin\BasinMaker"
#    ROOT = r"D:\Zelalem\CLRH_Basin\CLRH_Basin"  # for the raw data
    CRS = "EPSG:3979" #"EPSG:4326"  # "EPSG:3979" if desired

    layers = [
        "catchment_without_merging_lakes_v1-0.shp",
        "river_without_merging_lakes_v1-0.shp",
        "finalcat_info_v1-0.shp",
        "finalcat_info_riv_v1-0.shp",
        "sl_connected_lake_v1-0.shp",
        "sl_non_connected_lake_v1-0.shp",
        "outline.shp",
        "poi_v1-0.shp",
    ]

    print("STARTING BASINMAKER-COMPATIBLE MERGE")
    print("=" * 78)

    for layer in layers:
        merge_all_basinmaker_layers(
            root_dir=ROOT,
            layer_name=layer,
            target_crs=CRS
        )

    print("\nALL DONE")
    print(f"Merged shapefiles are in:\n  {Path(ROOT) / 'merged_shp'}")

In [3]:
## Authomate basinmaker basin and lake aggregation

# === Configuration ===
input_dir = '/scratch/zelalem/merged_shp/'


# Assume watershed_regions is a list, so pick the first region folder
#watershed_name = watershed_regions[0]

# Ensure directory exists
region_dir = os.path.join(input_dir)
#region_dir = os.path.join(output_dir, watershed_name)
os.makedirs(region_dir, exist_ok=True)

# Download files
wget.download(
    "https://github.com/dustming/RoutingTool/wiki/Files/landuse_info_routing.csv",
    out=region_dir
)
wget.download(
    "https://github.com/dustming/RoutingTool/wiki/Files/soil_info_routing.csv",
    out=region_dir
)
wget.download(
    "https://github.com/dustming/RoutingTool/wiki/Files/veg_info_routing.csv",
    out=region_dir
)

# Build paths to the downloaded files
path_veg_info = os.path.join(region_dir, "veg_info_routing.csv")
path_landuse_info = os.path.join(region_dir, "landuse_info_routing.csv")
path_soil_info = os.path.join(region_dir, "soil_info_routing.csv")

In [ ]:
# *TWO* USER INPUTs and the USER INPUT block denoted in lines below.  No changes required to run initially. Other user inputs here if this is adopted for something other than CLRH.
# Press play button to the left.  The code cell below MUST BE executed after Section 6 above is executed successfully.

#simplified HRU generation process to create only two HRU types: Land and Lake HRUs
input_dir = '/scratch/zelalem/merged_shp'

# define the input folder
input_routing_product_folder=input_dir  # <-- USER INPUT.  Must change if Section 6 cell not run.

# define another folder that will save the outputs
# watershed_name is a variable defined in section 3. Redefine here if necessary.
HRU_output_folder = os.path.join(input_routing_product_folder,'land_lake_HRUs')

# define version number of the routing product
# the version number of  CLRH is v1-0
version_number = 'v1-0' #

bm = basinmaker.postprocess()
start = time.time()
bm.Generate_HRUs(
    path_output_folder=HRU_output_folder, 
    path_subbasin_polygon        =  os.path.join(input_routing_product_folder, "finalcat_info_"+version_number+".shp"),
    path_landuse_polygon="#",
    path_soil_polygon   ="#",
    path_other_polygon_1="#",
    path_landuse_info=path_landuse_info,
    path_soil_info   =path_soil_info,
    path_veg_info    =path_veg_info,
    path_to_dem = "#", # # USER INPUT set to "#" at defaut abd indicates no DEM. For routing only mode, DEM is NOT needed. Likey sufficient in most semi-distributed models not in mountains.
                      # os.path.join(os.getcwd(),'02LE024','dem.tif'), # In order for HRUs to be assigned elevations not equal to subbasin avg elevations, DEM required.
    #path_to_dem =path_to_dem,
    area_ratio_thresholds = [0,0,0],     # use [0,0,0] for land and lake HRUs.  This is active input if building more complex HRUs. All zeros means keep all small HRUs.  See below ***
    gis_platform="purepy",
    projected_epsg_code = 'EPSG:3979',  # EPSG:3979 for CLRH   # OLRRPv2 - EPSG:3161 corresponds to the projected coordinate system NAD83 / Ontario MNR Lambert. Used for aspect/area calculation.
    pixel_size = 30  #90         # User-defined grid size in m. We recommend using 30 m for CLRH and OLRRP and 90 m for NA.
                                 # The unit follows the coordinate system of the routing network polygons.
)

### ***# Comment on area_ratio_thresholds = [0.1,0.1,0.1].  In BasinMaker HRU calculation, each layer will firstly be overlaid to the subbasin map.
# First fraction applies to first layer.  Each class in that layer covers a fraction of each subbasin (i.e., the classes area in subbasin over the
# the subbasin area) and if that is smaller than the defined threshold value, that class will then be dissolved
# into the largest part. For example, if forest area ratio in a subbasin, say subbasin #10,
# is 0.05, while we set the area threshold for land cover layer is 0.1, the forest polygons
# will then be dissolved to the largest land cover class in subbasin #10.

end = time.time()
print("This section took  ", end - start, " seconds")

In [ ]:
# 8.0 Produce Raven-required inputs

# One ***USER INPUT*** denoted below.  Look below before pressing play.

# NOTE: the code cell below MUST BE executed after the Generate_HRUs function code cell above is executed successfully.

# define another folder that will save the outputs
main_output_directory = '/scratch/zelalem/merged_shp/Mackenzie'
raven_model_dir = os.path.join(main_output_directory,'Raven_inputs')

bm = basinmaker.postprocess()

bm.Generate_Raven_Model_Inputs(
    path_hru_polygon         = os.path.join(HRU_output_folder, "finalcat_hru_info.shp"),
    model_name            ="Mackenzie",        # <-- ***USER INPUT***.  This is used for naming the output files, which are Raven model input files.
    subbasingroup_names_channel   =["Allsubbasins"],     # A subbasin group will be created in the rvh file for simultaneous manipulation in Raven modeling.
    subbasingroup_length_channel   =[-1],
    subbasingroup_name_lake      =["AllLakesubbasins"],
    subbasingroup_area_lake      =[-1],
    path_output_folder         = raven_model_dir,
    aspect_from_gis          = 'purepy',
)

# zip file
zipfile = os.path.join(os.path.dirname(raven_model_dir),os.path.basename(raven_model_dir) + '.zip')
!zip -q -r  "$zipfile" "$raven_model_dir"
print("The zipped Raven input files saved at ",os.path.basename(raven_model_dir) + '.zip')

In [ ]:
"""
7.0 Create HRUs

*TWO* USER INPUTs and the USER INPUT block denoted in lines below.  No changes required to run initially. Other user inputs here if this is adopted for something other than CLRH.
Press play button to the left.  The code cell below MUST BE executed after Section 6 above is executed successfully.

simplified HRU generation process to create only two HRU types: Land and Lake HRUs
"""
# ------------------------------------------------------------------
# User inputs
# ------------------------------------------------------------------
# Routing product version
version_number = 'v1-0'

# Watershed regions to process
# watershed_regions = ['Mackenzie']
# watershed_regions = [name for name in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, name))]

# ------------------------------------------------------------------
# Processing loop
# ------------------------------------------------------------------
for watershed_region in watershed_regions:

    input_routing_product_folder = input_dir

    HRU_output_folder = os.path.join(
        input_routing_product_folder, 'land_lake_HRUs'
    )

    try:
        print(f"Starting processing for {watershed_region}...")
        
        bm = basinmaker.postprocess()
        start = time.time()

        bm.Generate_HRUs(
            path_output_folder=HRU_output_folder,

            path_subbasin_polygon=os.path.join(
                input_routing_product_folder,
                f"finalcat_info_{version_number}.shp"
            ),

            path_landuse_polygon="#",
            path_soil_polygon="#",
            path_other_polygon_1="#",

            path_landuse_info=path_landuse_info,
            path_soil_info=path_soil_info,
            path_veg_info=path_veg_info,

            # DEM not required for routing-only mode
            path_to_dem="#",

            # Keep all HRUs (land + lake)
            area_ratio_thresholds=[0, 0, 0],

            gis_platform="purepy",

            # EPSG:3979 for CLRH
            projected_epsg_code=3979,

            # Grid size in meters
            pixel_size=30
        )

        end = time.time()
        print(f"This section took {end - start:.2f} seconds")
        print(f"Processing done for network: {watershed_region}")

    except Exception as e:
        print(f"Error processing {watershed_region}: {e}")

In [ ]:
"""
8.0 Produce Raven-required inputs
One ***USER INPUT*** denoted below.  Look below before pressing play.
NOTE: the code cell below MUST BE executed after the Generate_HRUs function code cell above is executed successfully.

"""
import zipfile as zipmod   # <-- alias avoids name collisions

input_dir = r'D:\Zelalem\CLRH_Basin\BasinMaker'
# watershed_regions = ['01_01']
# watershed_regions = [name for name in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, name))]

# ------------------------------------------------------------------
# Processing loop
# ------------------------------------------------------------------
for watershed_region in watershed_regions:

    try:
        print(f"Generating Raven inputs for {watershed_region}...")

        main_output_directory = os.path.join(
            input_dir, watershed_region, 'increase_drainage_area'
        )

        HRU_output_folder = os.path.join(
            main_output_directory, 'land_lake_HRUs'
        )

        raven_model_dir = os.path.join(
            main_output_directory, 'Raven_inputs'
        )

        bm = basinmaker.postprocess()

        bm.Generate_Raven_Model_Inputs(
            path_hru_polygon=os.path.join(
                HRU_output_folder, "finalcat_hru_info.shp"
            ),
            model_name=watershed_region,
            subbasingroup_names_channel=["Allsubbasins"],
            subbasingroup_length_channel=[-1],
            subbasingroup_name_lake=["AllLakesubbasins"],
            subbasingroup_area_lake=[-1],
            path_output_folder=raven_model_dir,
            aspect_from_gis='purepy',
        )

        # ----------------------------------------------------------
        # Zip Raven inputs (SAFE version)
        # ----------------------------------------------------------
        zip_path = os.path.join(
            main_output_directory,
            os.path.basename(raven_model_dir) + ".zip"
        )

        with zipmod.ZipFile(zip_path, 'w', zipmod.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(raven_model_dir):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, raven_model_dir)
                    zf.write(full_path, arcname)

        print(f"Zipped Raven input files saved at {zip_path}")
        print(f"Processing done for network: {watershed_region}")

    except Exception as e:
        print(f"Error processing {watershed_region}: {e}")

In [ ]:
# *TWO* USER INPUTs and the USER INPUT block denoted in lines below.  No changes required to run initially. Other user inputs here if this is adopted for something other than CLRH.
# Press play button to the left.  The code cell below MUST BE executed after Section 6 above is executed successfully.

#simplified HRU generation process to create only two HRU types: Land and Lake HRUs

# define the input folder
input_routing_product_folder=folder_product_after_increase_catchment_drainage_area  # <-- USER INPUT.  Must change if Section 6 cell not run.

# define another folder that will save the outputs
# watershed_name is a variable defined in section 3. Redefine here if necessary.
HRU_output_folder = os.path.join(main_output_directory,watershed_name,network_name,'land_lake_HRUs')

# define version number of the routing product
# the version number of  CLRH is v1-0
version_number = 'v1-0' #

bm = basinmaker.postprocess()
start = time.time()
bm.Generate_HRUs(
    path_output_folder=HRU_output_folder,
    path_subbasin_polygon        =  os.path.join(input_routing_product_folder, "finalcat_info_"+version_number+".shp"),
    path_landuse_polygon="#",
    path_soil_polygon   ="#",
    path_other_polygon_1="#",
    path_landuse_info=path_landuse_info,
    path_soil_info   =path_soil_info,
    path_veg_info    =path_veg_info,
    path_to_dem = "#", # # USER INPUT set to "#" at defaut abd indicates no DEM. For routing only mode, DEM is NOT needed. Likey sufficient in most semi-distributed models not in mountains.
                      # os.path.join(os.getcwd(),'02LE024','dem.tif'), # In order for HRUs to be assigned elevations not equal to subbasin avg elevations, DEM required.
    #path_to_dem =path_to_dem,
    area_ratio_thresholds = [0,0,0],     # use [0,0,0] for land and lake HRUs.  This is active input if building more complex HRUs. All zeros means keep all small HRUs.  See below ***
    gis_platform="purepy",
    projected_epsg_code = 'EPSG:3979',  # EPSG:3979 for CLRH   # OLRRPv2 - EPSG:3161 corresponds to the projected coordinate system NAD83 / Ontario MNR Lambert. Used for aspect/area calculation.
    pixel_size = 30  #90         # User-defined grid size in m. We recommend using 30 m for CLRH and OLRRP and 90 m for NA.
                                 # The unit follows the coordinate system of the routing network polygons.
)

### ***# Comment on area_ratio_thresholds = [0.1,0.1,0.1].  In BasinMaker HRU calculation, each layer will firstly be overlaid to the subbasin map.
# First fraction applies to first layer.  Each class in that layer covers a fraction of each subbasin (i.e., the classes area in subbasin over the
# the subbasin area) and if that is smaller than the defined threshold value, that class will then be dissolved
# into the largest part. For example, if forest area ratio in a subbasin, say subbasin #10,
# is 0.05, while we set the area threshold for land cover layer is 0.1, the forest polygons
# will then be dissolved to the largest land cover class in subbasin #10.

end = time.time()
print("This section took  ", end - start, " seconds")

In [ ]:
# 8.0 Produce Raven-required inputs

# One ***USER INPUT*** denoted below.  Look below before pressing play.

# NOTE: the code cell below MUST BE executed after the Generate_HRUs function code cell above is executed successfully.

# define another folder that will save the outputs
main_output_directory = r'D:\Zelalem\CLRH_Basin\BasinMaker\01_01\increase_drainage_area'
raven_model_dir = os.path.join(main_output_directory,'Raven_inputs')

bm = basinmaker.postprocess()

bm.Generate_Raven_Model_Inputs(
    path_hru_polygon         = os.path.join(HRU_output_folder, "finalcat_hru_info.shp"),
    model_name            ="01_01",        # <-- ***USER INPUT***.  This is used for naming the output files, which are Raven model input files.
    subbasingroup_names_channel   =["Allsubbasins"],     # A subbasin group will be created in the rvh file for simultaneous manipulation in Raven modeling.
    subbasingroup_length_channel   =[-1],
    subbasingroup_name_lake      =["AllLakesubbasins"],
    subbasingroup_area_lake      =[-1],
    path_output_folder         = raven_model_dir,
    aspect_from_gis          = 'purepy',
)

# zip file
zipfile = os.path.join(os.path.dirname(raven_model_dir),os.path.basename(raven_model_dir) + '.zip')
!zip -q -r  "$zipfile" "$raven_model_dir"
print("The zipped Raven input files saved at ",os.path.basename(raven_model_dir) + '.zip')

# **2.0 Upload a routing network into your Colab session**

In this step, we demonstrate how you can move an existing routing network/hydrofabric into this Colab environment.  There are three options for section 2 below:

      a) Download a gauge level Watershed routing network from CLRH website
      b) Download a regional level routing network from CLRH website
      c) Upload your CLRH-based routing network from another location

Consider if you want to run the code cell below. Read code cell comments before clicking play.


In [ ]:
# OPTIONAL
# Mount your Google Drive and enable easy file transfer to and from Colab to your 'My Drive' folder.

# Ultimately you can easily download zip file BasinMaker outputs without mounting your Google Drive and htis is perhaps
# just a bit faster output file access.

# No user inputs, just press play button to the left.

from google.colab import drive
drive.mount('/content/drive')

Go to the section below for the option you want to choose and do not run the code cells in the other two options.

## **2(a). Download a GAUGE level watershed routing network from CLRH website**





Use this if you know the  hydrometric gauge name (the 'Obs_NM' attribute) you want the CLRH routing network for (or the routing network of any CLRH POI type).

Routing networks accessed using this method do not retain upstream POI layer info (an oversight to be corrected).  But note the upstream POI locations are retained and reflected in Raven .rvh files if you build a Raven model in Section 8.0. If you need a POI shapefile including all upstream POI, access the hydrometric station using option 2(b) below.

In [ ]:
# One ***USER INPUT*** BELOW to look at before you press the play button.

# This download option uses hydrometric gauge name to download the routing product watershed.
# Note the longest this should take is ~70 seconds [e.g., if you choose the largest possible CLRH gauge on the McKenzie]

# define the product name
product_name = 'CLRH'  # In this notebook, we exclusively use the Canadian Lake and River Hydrofabric

# define the gauge name
gauge_name= '07BK003'  # ***USER INPUT***. # gauge_name= '10ND005'.  This is the 'Obs_NM' attribute in the CLRH POI layer which is
                       # the point of interest ID. For hydrometric station POI, this is the monitoring agency assigned station ID.

network_name = gauge_name # for easy use in Section 3.0

SubID,product_path = Download_Routing_Product_For_One_Gauge(gauge_name = gauge_name,product_name = product_name)
SubID = [SubID]
download_directory_name = "CLRH_download_" + gauge_name
download_directory_path = os.path.join(os.getcwd(),download_directory_name)
os.rename(product_path, download_directory_path)
os.remove(gauge_name + '.zip')

print('... and was just moved to temporary directory called ', download_directory_path)
print('SubID is ', SubID)
#SubID above is -1, not assigned

Now skip down to Section 3.0.

## **2(b). Download a REGIONAL level routing network from CLRH website**

Skip to Section 3.0 if you did Section 2(a) above.

One reason you may want to access a WSC gauge (more generally a CLRH POI location) using this option is that accessing a gauge this way retains all upstream POI in the POI layer.  Upstream gauges of WSC gauges accessed using Section 2(a) do not retain upstream POI layer info (an oversight to be corrected).  

Comments in code cell give guidance how to change inputs ...

In [ ]:
# THREE ***USER INPUTs*** required below to look at before you press the play button.

# Import large regional file from CLRH:
# Note the longest this should take is ~70 seconds [e.g., if you choose the largest possible CLRH gauge on the McKenzie]

region_name = 'Mackenzie'   # ***USER INPUT***

zipfile_name = region_name + ".zip"
url = "https://hydrology.uwaterloo.ca/CLRH/SSDAdata/" + zipfile_name
filename = wget.download(url)   # downloads file to content/ directory level
print(f'File downloaded and saved to: {filename}')  # zip in content level directory
print(f'')

zipfile_path="/content/"+zipfile_name
unzipfile_path="/content/upload_" + region_name
!unzip "$zipfile_path" -d "$unzipfile_path"

download_directory_path = unzipfile_path + "/" + region_name

network_name = 'Mackenzie' # ***USER INPUT***.  The name you want for the network you are creating.  Corresponding to the SubID location below.
                        # Does not have to be a gauge location.

SubID = [-1] # keep this as is here.

# Click the Folder icon to the far left to inspect the directory structure you created for this Colab session (these are temporary and disappears once session ends)

*DEVELOPER TO DO:  to provide users help w a temporary map using lat-lon for selecting a SubID that is not associated with a gauge.  GIS savy folks won't need this help.*

You should see 'extracting' messages in above code cell output without error after pressing play. If so, your routing network is now uploaded to your Colab session.

Now skip down to Section 3.0.

## **2(c). Upload your CLRH-based routing network from another location**

This option is what you would want to use if you have a CLRH-based routing network saved in another location that you want to post-process with Basinmaker.

Skip to Section 3.0 if you did Section 2(a) or 2(b) above.  

This is for users to upload a CLRH network they have already accessed and saved locally into the Colab environment for further post-processing. This example notebook is handy if you have initially done some post-processing to a routing network from CLRH, saved the result and then decided you want to simplify the routing network discretization evern further in a new BasinMaker session.


In [ ]:
# Just press play button and then follow instructions to upload a zip file into Colab content folder.


# Import a single zipfile from your PC.  e.g., assume you import file called 02E_02F.zip
from google.colab import files
uploaded = files.upload()
# Comment. After clicking play above option requires you to Click the 'Choose Files' button below and select ***.zip from your PC


Based on the uploaded filename above change input below.

In [ ]:
# TWO USER INPUTS in code below.

# The code below and then also in Section 3.0 assumes your routing network shapefiles will all be contained
# within a single directory, no subdirectories, called 'your_file_name' once they are unzipped.  If this is not the case,
# you need to modify code below somehow OR change zip file contents accordingly.

your_file_name = '02G'   # ***USER INPUT***.  No .zip extenstion here!

zipfile_name = your_file_name + ".zip"

zipfile_path="/content/"+zipfile_name
unzipfile_path="/content/upload_" + your_file_name
!unzip "$zipfile_path" -d "$unzipfile_path"

download_directory_path = unzipfile_path + "/" + your_file_name

network_name = '02G' # ***USER INPUT***.  The name you want for the network you are creating.  Corresponding to the SubID location below.
                        # Does not have to be a gauge location.

SubID = [-1] # keep this as is here.

# Click the Folder icon to the far left to inspect the directory structure you created for this Colab session (these are temporary and disappears once session ends)

You should see 'extracting' messages in above code cell without error after pressing play.  If so, your routing network/product is now uploaded to your Colab session.

# **3.0 Extract drainage area from uploaded routing network**

In [ ]:
# ONE ***USER INPUT*** you should look at below to look at before you press the play button.

# default here is to assume you want entire routing network just uploaded to Colab OR you have already specified SubID above in Section 2(b) or 2(c).

# If you want to only extract any user specified subwatershed of the network currently loaded in Colab, uncomment below line and enter the SubID
# corresponding to your subwatershed outlet of interest.

# SubID = [28002037] # OPTIONAL ***USER INPUT***   # Default is to leave this commented out for all access options above (2a,b,c).
#  SubID = [28002037,28002038,28002040] is the format you use to extract multiple watersheds into the same shapefile
#  from your uploaded routing network.  For example to grab the all the subbasins flowing into Lake Erie.

# NOTE: You can find SubID of all subbasins in the finalcat_info_v1-0.dbf file ... or looking at shapefile of this CLRH layer.
# For any *watershed* routing network you might upload, SubID of an outlet draining out of the network is determined from finalcat_info_v1-0.dbf
# where DowSubId=-1.

 # NOTE: a new download is required in Step 2 if you ran this code cell with SubID=-1 already in this Colab session.

watershed_name = network_name  # Use Section 2 defined variable

main_output_directory=os.path.join(os.getcwd(),watershed_name)
os.makedirs(main_output_directory, exist_ok=True)
folder_product_for_interested_gauges=os.path.join(os.getcwd(),watershed_name,'extraction')

############
if SubID == [-1]:
    shutil.move(download_directory_path, main_output_directory)
    os.rename(os.path.join(main_output_directory,os.path.basename(download_directory_path)), folder_product_for_interested_gauges)
    print("Extracted routing network now located in ", folder_product_for_interested_gauges)
else:
    #extracting network for subregion upstream of region/watershed outlet
    subid_of_interested_gauges= SubID  # this is variable from 2b-3 code cell
    # Initialize the basinmaker
    start = time.time()
    bm = basinmaker.postprocess()
    # extract subregion of the routing product
    bm.Select_Subregion_Of_Routing_Structure(
        path_output_folder = folder_product_for_interested_gauges,
        routing_product_folder = download_directory_path,
        most_down_stream_subbasin_ids=subid_of_interested_gauges,
        most_up_stream_subbasin_ids=[-1],     # -1: extract to the most-upstream (headwater) subbasin; other subbasin ID: extract the areas from the outlet to the provided subbasin.
        gis_platform="purepy",
    )
    end = time.time()
    shutil.rmtree(download_directory_path)
    print("This section took  ", end - start, " seconds")
    print("Extracted routing network now located in ", folder_product_for_interested_gauges)
############


The output of this function is the set of eight GIS shapefiles as described in the CLRH attribute tables file available on the CLRH website.  The key shapefiles are as follows in short:

* finalcat_info_v1-0. : subbasin polygons respecting lakes (all subbasins in the figure below comes from this file)
* finalcat_info_riv_v1-0. : river network polylines in each subbasin polygon
* poi_v1-0. : points of interest in the hydrofabric (e,g., hydrometric gauges)
* sl_connected_lake. : the lake polygons of lakes that are connected by the finalcat_info_riv.shp (these connect downstream via an explicit river channel in the hydrofabric).  This only exists if such lakes are present in the extracted routing network.
* sl_non_connected_lake. : the lake polygons of lakes that are not connected by the finalcat_info_riv.shp (these connect downstream implicitly, to the next subbasin without an explicit river channel in the hydrofabric). This only exists if such lakes are present in the extracted routing network.

Have a look in the 'extraction' folder to the left to find these outputs.  Note the next code cell visualizes the result and zips up the shapefiles for easy export.


**Plot and zip the extracted routing product (The plot function may not work for a large watershed, e.g. 50,000 km2 or more)**

As done above, we again use function `plot_routing_product_with_ipyleaflet` to visualize the extracted routing product. The input of this function is the path to the routing product folder and the routing product version number.

You can download the zip file of the extracted watershed after running the following section of the code, since the resultant folder is compressed.

***Please note this notebook produces outputs on temporary drive and you should download zip outputs created before closing the Colab website***

In [ ]:
# No user inputs required.  Press play button to the left

#Define the path to the routing product folder
path_to_input_routing_product_folder = folder_product_for_interested_gauges # example Path_to_input_routing_product_folder = folder_product_for_interested_gauges

# Define the version number
# The version number of  Canadian Lake and River Hydrofabric is v1-0

routing_product_version_number = 'v1-0' # this specifies CLRH version 1.0.

# zip file
zipfile = os.path.join(os.path.dirname(path_to_input_routing_product_folder),os.path.basename(path_to_input_routing_product_folder) + '.zip')
!zip -r -q "$zipfile" "$path_to_input_routing_product_folder"
print("The zipped routing product was saved at ",os.path.basename(path_to_input_routing_product_folder) + '.zip')

# plot product
# plot_routing_product_with_ipyleaflet(path_to_product_folder = path_to_input_routing_product_folder,version_number = routing_product_version_number)


# **4.0 Use RavenView website to check routing network BasinMaker produced**

*If you extracted above using a SubID = -1, your RavenView files are not yet available.  Complete Section 5.0 and then come back here. Then you can find the RavenView files below instead in the Section 5.0 output folder.*

Now, let's inspect some of the files produced for the watershed/gauge you just extracted. Download two files from '"/"content/05CB006/extraction', to your computer.

The two BasinMaker/CLRH output files from the extraction step need to be  uploaded to RavenView:
- **"finalcat_info_v2-1.geojson"**
- **"routing_product_lake_river.geojson"**

Note also that Ravenview compliant files are always created after STEP 5, 6, 7 and 8.

**We will inspect the network topology using the above files viewed in RavenView online software here:**
http://raven.uwaterloo.ca/RavenView/RavenView.html

Go to RavenView website, click the "Import subbasin map file" button, and upload the "finalcat_info_v2-1.geojson" you now have.

Then click "Import river map" button to upload "routing_product_lake_river.geojson" you now have.

Now have a tour on RavenView. Click every button there to find out some interesting functionalities!

# **5.0 Simplify the routing product by filtering lakes**

Here we specify which lakes to filter (remove) from the extracted routing network. The code cell below MUST BE executed after Section 3 above is executed successfully. Below you get to filter out lakes below a lake surface area threshold (separate thresholds for coneected and non-connected lakes) and in addition have an option to include special lakes even if they are below the thresholds.

After you run through all post-processing Sections under default USER INPUT settings, we recommend you come back to the code cell below and try alternative USER INPUT settings.

In [ ]:
folder_product_after_filter_lakes = os.path.join(folder_product_for_interested_gauges,'filter_lakes')
folder_product_after_filter_lakes

In [ ]:
# Please note the *three* USER INPUT denoted lines below.  No changes required to run initially.
# Press play button to the left. The code cell below MUST BE executed after Section 3 above is executed successfully.

#network_name = 'Mackenzie'
network_name = '02H'
watershed_name = network_name  # Use Section 2 defined variable
main_output_directory= r'D:\CLRH_Basin\CLRH_Basin'
os.makedirs(main_output_directory, exist_ok=True)
folder_product_for_interested_gauges=os.path.join(main_output_directory,watershed_name,network_name)

# define the input product folder path which is the output folder of previous section
input_routing_product_folder=folder_product_for_interested_gauges


# USER INPUT --> define a list containing HyLakeId ID of lakes of interest that are
# NOT to be removed even if their area is smaller than lake area threshold.
# Where to find out those IDs of lakes:
#   The finalcat_info.shp has an attribute of lake IDs (named "HyLakeId").
# if you do not have interested lakes in mind to require network inclusion, use: "[]"
interested_lake_ids = []       # []    # <-- USER INPUT.  example format of interested_lake_ids = [108494,111111]

# define another folder that will save the outputs
# the watershed_name is defined in previous section.
folder_product_after_filter_lakes = os.path.join(input_routing_product_folder,'filter_lakes')

start = time.time()
bm = basinmaker.postprocess()

# remove small lakes
bm.Remove_Small_Lakes(
    path_output_folder = folder_product_after_filter_lakes,
    routing_product_folder = input_routing_product_folder,
    connected_lake_area_thresthold= 5,         # USER INPUT.  unit km2, this is to remove lakes with area <= threshold
    non_connected_lake_area_thresthold= 5,     # USER INPUT, this is to remove lakes with area  (= threshold
    selected_lake_ids=interested_lake_ids,
    gis_platform="purepy",
)
end = time.time()
print("This section took  ", end - start, " seconds")


# Please note the *one* USER INPUT denoted line below.  No changes required to run initially in tutorial.
# Press play button to the left.  The code cell below MUST BE executed after Section 5 above is executed successfully

# define the input folder path which is by default the output folder of section 5.
input_routing_product_folder=folder_product_after_filter_lakes
#input_routing_product_folder=folder_product_for_interested_gauges  # the directory path if you did not run code cells in section 5.

# define another folder that will save the outputs
folder_product_after_increase_catchment_drainage_area = os.path.join(main_output_directory,watershed_name,network_name,'increase_drainage_area')

# Initialize the basinmaker
start = time.time()
bm = basinmaker.postprocess()

# remove river reaches and increase size of subbasin
bm.Decrease_River_Network_Resolution(
    path_output_folder = folder_product_after_increase_catchment_drainage_area,
    routing_product_folder = input_routing_product_folder,
    minimum_subbasin_drainage_area= 100,    # <-- USER INPUT unit in km2. Definition of threshold is non-lake subbasins (and river reaches) with drainage areas < value, will be removed.
    #  For a subbasin removed, it will be merged to another subbasin to get a larger subbasin that meets the drainage area threshold.
    gis_platform="purepy",
)
end = time.time()
print("This section took  ", end - start, " seconds")

The output of this function is the set of eight GIS shapefiles as described in the CLRH attribute tables file available on the CLRH website. The key shapefiles are as follows in short:

- finalcat_info_v1-0. : subbasin polygons respecting lakes (all subbasins in the figure below comes from this file)
- finalcat_info_riv_v1-0. : river network polylines in each subbasin polygon
- poi_v1-0. : points of interest in the hydrofabric (e,g., hydrometric gauges)
- sl_connected_lake. : the lake polygons of lakes that are connected by the finalcat_info_riv.shp (these connect downstream via an explicit river channel in the hydrofabric). This only exists if such lakes are present in the extracted routing network.
- sl_non_connected_lake. : the lake polygons of lakes that are not connected by the finalcat_info_riv.shp (these connect downstream implicitly, to the next subbasin without an explicit river channel in the hydrofabric). This only exists if such lakes are present in the extracted routing network.

Have a look in the 'filter_lakes' folder to the left to find these outputs. Note the next code cell visualizes the result and zips up the shapefiles for easy export.

**Plot and zip the simplified routing product by removing lakes (the plot function may not work for large watershed)**

In [ ]:
# No user inputs required.  Press play button to the left. The code cell below MUST BE executed after Section 5 above is executed successfully.

# define the input product folder path which is by default the output folder of previous section
path_to_input_routing_product_folder = folder_product_after_filter_lakes
# path_to_input_routing_product_folder = folder_product_for_interested_gauges  # alternate INPUT if Section 5 skipped

# Define the version number
# The version number of  Canadian Lake and River Hydrofabric is v1-0

routing_product_version_number = 'v1-0' # v1-0 specifies CLRH

# zip file
zipfile = os.path.join(os.path.dirname(path_to_input_routing_product_folder),os.path.basename(path_to_input_routing_product_folder) + '.zip')
!zip -q -r  "$zipfile" "$path_to_input_routing_product_folder"
print("The zipped routing product was saved at ",os.path.basename(path_to_input_routing_product_folder) + '.zip')

# plot product
# plot_routing_product_with_ipyleaflet(path_to_product_folder = path_to_input_routing_product_folder,version_number = routing_product_version_number)


In [ ]:
# define the input folder path which is by default the output folder of section 5.
input_routing_product_folder=folder_product_after_filter_lakes
#input_routing_product_folder=folder_product_for_interested_gauges  # the directory path if you did not run code cells in section 5.

# define another folder that will save the outputs
folder_product_after_increase_catchment_drainage_area = os.path.join(main_output_directory,watershed_name,network_name,'increase_drainage_area')
folder_product_after_increase_catchment_drainage_area

# **6.0 Simplify the routing product by increasing size of subbasins**

Here we specify how much to remove (aggregate) small non-lake subbasins together to simplify the network. The code cell below MUST BE executed after Section 5 above is executed successfully. In the event you do not want to filter out any lakes in Section 5 above, simply run filter lakes with a tiny lake area threshold (e.g. 0.01) which will function not to remove any lakes but just instead produce the files expected for this Step.

Function below combines small non-lake subbasins less than an area threshold with other non-lake subbasins. This simplificantion step increases the average subbasin size and thus reduces the number of subbasins in your routing network.  Importantly, it has no impact on lake subbasins.

The BasinMaker function `Decrease_River_Network_Resolution`  will be used for this purpose. This function will merge any upstream subbasin with a drainage area at the outlet that is smaller than a given threshold with their downstream subbasin. The function starts in headwater subbasin and moves downstream and it does this throughout the domain being processed.

After you run through all 7 post-processing steps under default USER INPUT settings, we recommend you come back to the code cell below and try alternative USER INPUT settings.

More detailed description of function `Decrease_River_Network_Resolution` can be found in [here](https://basinmaker.readthedocs.io/en/latest/basinmaker_tools.html#increase-catchment-area).

In [ ]:
# Please note the *one* USER INPUT denoted line below.  No changes required to run initially in tutorial.
# Press play button to the left.  The code cell below MUST BE executed after Section 5 above is executed successfully

# define the input folder path which is by default the output folder of section 5.
input_routing_product_folder=folder_product_after_filter_lakes
#input_routing_product_folder=folder_product_for_interested_gauges  # the directory path if you did not run code cells in section 5.

# define another folder that will save the outputs
folder_product_after_increase_catchment_drainage_area = os.path.join(main_output_directory,watershed_name,network_name,'increase_drainage_area')

# Initialize the basinmaker
start = time.time()
bm = basinmaker.postprocess()

# remove river reaches and increase size of subbasin
bm.Decrease_River_Network_Resolution(
    path_output_folder = folder_product_after_increase_catchment_drainage_area,
    routing_product_folder = input_routing_product_folder,
    minimum_subbasin_drainage_area= 100,    # <-- USER INPUT unit in km2. Definition of threshold is non-lake subbasins (and river reaches) with drainage areas < value, will be removed.
    #  For a subbasin removed, it will be merged to another subbasin to get a larger subbasin that meets the drainage area threshold.
    gis_platform="purepy",
)
end = time.time()
print("This section took  ", end - start, " seconds")


The output of this function is the set of eight GIS shapefiles as described in the CLRH attribute tables (see CLRH data specifications file) available on the CLRH website. The key shapefiles are as follows in short:

- finalcat_info_v1-0. : subbasin polygons respecting lakes (all subbasins in the figure below comes from this file)
- finalcat_info_riv_v1-0. : river network polylines in each subbasin polygon
- poi_v1-0. : points of interest in the hydrofabric (e,g., hydrometric gauges)
- sl_connected_lake. : the lake polygons of lakes that are connected by the finalcat_info_riv.shp (these connect downstream via an explicit river channel in the hydrofabric). This only exists if such lakes are present in the extracted routing network.
- sl_non_connected_lake. : the lake polygons of lakes that are not connected by the finalcat_info_riv.shp (these connect downstream implicitly, to the next subbasin without an explicit river channel in the hydrofabric). This only exists if such lakes are present in the extracted routing network.

Have a look in the 'increase_drainage_area' folder to the left to find these outputs. Note the next code cell visualizes the result and zips up the shapefiles for easy export.

**Plot and zip the simplified routing product by increasing subbasin size (the plot function may not work for large watershed)**





In [ ]:
# No user inputs required.  Press play button to the left

path_to_input_routing_product_folder = folder_product_after_increase_catchment_drainage_area # example Path_to_input_routing_product_folder = folder_product_after_increase_catchment_drainage_area

# Define the version number
# The version number of  Canadian Lake and River Hydrofabric is v1-0

routing_product_version_number = 'v1-0' # specifies CLRH

# zip file
zipfile = os.path.join(os.path.dirname(path_to_input_routing_product_folder),os.path.basename(path_to_input_routing_product_folder) + '.zip')
!zip -q -r  "$zipfile" "$path_to_input_routing_product_folder"
print("The zipped routing product was saved at ",os.path.basename(path_to_input_routing_product_folder) + '.zip')

# plot product
# plot_routing_product_with_ipyleaflet(path_to_product_folder = path_to_input_routing_product_folder,version_number = routing_product_version_number)


# **7.0 Create HRUs**

Many semi-distributed/distributed hydrological models are built on the concept of the hydrological response unit (HRU). A HRU is the smallest response unit containing unique geo-spatial information. Thus, defining HRUs is the process to overlay and union different geo-spatial data layers, such as land cover and/or soil, and/or elevation band.

The BasinMaker function `Generate_HRUs` will be used for this purpose and can create HRUs using anywhere from 0 to 3 GIS layers with your delineated subbasins. A more detailed description of the function `Generate_HRUs` can be found [here](https://basinmaker.readthedocs.io/en/latest/basinmaker_tools.html).

The simplest HRU approach some semi-distributed models such as Raven can utilize is to define only two types of HRUs:  land HRUs and lake HRUs.  All non-lake land within a subbasin defines a single 'land' HRU. This is the required HRU discretization when running the Raven Hydrological model in routing-only mode. Provided with runoff/recharge fluxes, Raven is able to build a routing-only model to route these fluxes to the outlet. We just need an HRU map that distinguishes land and lake HRUs.

In this section, we will build the simplest set of land and lake only HRUs. The best link with an example for building complex HRUs is actually the public OLRRPv2 Colab notebook (https://colab.research.google.com/drive/1tsSUiVXmU1VBR_4U313Tin_m2zmFDzRP?usp=sharing). The OLRRPv2 Colab site provides users instructions and additional geospatial data for building more complex HRUs within each subbasin.  


## Define only LAND and LAKE HRUs

**In this example, we define the simplest two HRU types and these require you upload three small csv files to the Colab path:** e.g. if your watershed_name=05CB006 then this path is content/05CB006/.  The code cell below does this upload for you.

*   landuse_info_routing.csv
*   soil_info_routing.csv
*   veg_info_routing.csv  

In [ ]:
wget.download("https://github.com/dustming/RoutingTool/wiki/Files/landuse_info_routing.csv", out=os.path.join(main_output_directory,watershed_name))
wget.download("https://github.com/dustming/RoutingTool/wiki/Files/soil_info_routing.csv", out=os.path.join(main_output_directory,watershed_name))
wget.download("https://github.com/dustming/RoutingTool/wiki/Files/veg_info_routing.csv", out=os.path.join(main_output_directory,watershed_name))
path_veg_info = os.path.join(main_output_directory, watershed_name, "veg_info_routing.csv")
path_landuse_info = os.path.join(main_output_directory, watershed_name, "landuse_info_routing.csv")
path_soil_info = os.path.join(main_output_directory, watershed_name, "soil_info_routing.csv")

Land versus lake HRUs are the simplest way to define HRUs.  All land HRUs are treated the same (e.g., implictly assumed when writing Raven input files in next Step to have the same generic landcover and soil). As such, the content of these three csv files have  placeholder/dummy inputs so the BasinMaker HRU creation function runs properly.  

Optionally, users can upload their preferred DEM as well using the code cell below.  If users do not upload their preferred DEM, then subbasin/HRU attributes like average elevation, slope, aspect etc. are assigned from CLRH (where CLRH used a 30 m continental DEM for subbasin attributes).  Otherwise, the subbasin and HRU attributes are all recalculated from user uploaded DEM.

In [ ]:
# Optional: upload user specific DEM covering the routing network you have. Must 100% cover the entire routing network otherwise errors.
# UNCOMMENT everything below, change inputs to your case case study DEM if you have the DEM. Note the MRDEM_RedDeer_masked.tif is NOT provided.

# source_file = '/content/drive/MyDrive/MRDEM_RedDeer_masked.tif'  # <-- USER INPUT
# destination_file = '/content/MRDEM_RedDeer_masked.tif'           # <-- USER INPUT
# shutil.copy(source_file, destination_file)  # moves file into current Colab session

# path_to_dem=os.path.join(os.getcwd(),'MRDEM_RedDeer_masked.tif') # <-- USER INPUT

-

In [ ]:
# *TWO* USER INPUTs and the USER INPUT block denoted in lines below.  No changes required to run initially. Other user inputs here if this is adopted for something other than CLRH.
# Press play button to the left.  The code cell below MUST BE executed after Section 6 above is executed successfully.

#simplified HRU generation process to create only two HRU types: Land and Lake HRUs

# define the input folder
input_routing_product_folder=folder_product_after_increase_catchment_drainage_area  # <-- USER INPUT.  Must change if Section 6 cell not run.

# define another folder that will save the outputs
# watershed_name is a variable defined in section 3. Redefine here if necessary.
HRU_output_folder = os.path.join(main_output_directory,watershed_name,network_name,'land_lake_HRUs')

# define version number of the routing product
# the version number of  CLRH is v1-0
version_number = 'v1-0' #

bm = basinmaker.postprocess()
start = time.time()
bm.Generate_HRUs(
    path_output_folder=HRU_output_folder,
    path_subbasin_polygon        =  os.path.join(input_routing_product_folder, "finalcat_info_"+version_number+".shp"),
    path_landuse_polygon="#",
    path_soil_polygon   ="#",
    path_other_polygon_1="#",
    path_landuse_info=path_landuse_info,
    path_soil_info   =path_soil_info,
    path_veg_info    =path_veg_info,
    path_to_dem = "#", # # USER INPUT set to "#" at defaut abd indicates no DEM. For routing only mode, DEM is NOT needed. Likey sufficient in most semi-distributed models not in mountains.
                      # os.path.join(os.getcwd(),'02LE024','dem.tif'), # In order for HRUs to be assigned elevations not equal to subbasin avg elevations, DEM required.
    #path_to_dem =path_to_dem,
    area_ratio_thresholds = [0,0,0],     # use [0,0,0] for land and lake HRUs.  This is active input if building more complex HRUs. All zeros means keep all small HRUs.  See below ***
    gis_platform="purepy",
    projected_epsg_code = 'EPSG:3979',  # EPSG:3979 for CLRH   # OLRRPv2 - EPSG:3161 corresponds to the projected coordinate system NAD83 / Ontario MNR Lambert. Used for aspect/area calculation.
    pixel_size = 30  #90         # User-defined grid size in m. We recommend using 30 m for CLRH and OLRRP and 90 m for NA.
                                 # The unit follows the coordinate system of the routing network polygons.
)

### ***# Comment on area_ratio_thresholds = [0.1,0.1,0.1].  In BasinMaker HRU calculation, each layer will firstly be overlaid to the subbasin map.
# First fraction applies to first layer.  Each class in that layer covers a fraction of each subbasin (i.e., the classes area in subbasin over the
# the subbasin area) and if that is smaller than the defined threshold value, that class will then be dissolved
# into the largest part. For example, if forest area ratio in a subbasin, say subbasin #10,
# is 0.05, while we set the area threshold for land cover layer is 0.1, the forest polygons
# will then be dissolved to the largest land cover class in subbasin #10.

end = time.time()
print("This section took  ", end - start, " seconds")

#### Plot and Zip LAKE and LAND HRUs

In [ ]:
# Press play button to the left. The code cell below MUST BE executed after above code cell above is executed successfully.

hru_polygon = geopandas.read_file(os.path.join(HRU_output_folder, "finalcat_hru_info.shp"))
#ax = hru_polygon.plot(figsize=(24, 24),linewidth = 1,edgecolor='black',facecolor="none",zorder=0)

if os.path.exists(os.path.join(input_routing_product_folder, "sl_connected_lake_"+version_number+".shp")):
  sl_lake_ply = geopandas.read_file(os.path.join(input_routing_product_folder, "sl_connected_lake_"+version_number+".shp")).to_crs(hru_polygon.crs)
  #sl_lake_ply.plot(ax = ax, linewidth = 0.00001,edgecolor='black',alpha=0.6,zorder=1)

if os.path.exists(os.path.join(input_routing_product_folder, "sl_non_connected_lake_"+version_number+".shp")):
  nsl_lake_ply = geopandas.read_file(os.path.join(input_routing_product_folder, "sl_non_connected_lake_"+version_number+".shp")).to_crs(hru_polygon.crs)
  #nsl_lake_ply.plot(ax = ax, linewidth = 0.00001,edgecolor='black',alpha=0.6,zorder=1)

# zip file
zipfile = os.path.join(os.path.dirname(HRU_output_folder),os.path.basename(HRU_output_folder) + '.zip')
!zip -q -r  "$zipfile" "$HRU_output_folder"
print("The zipped HRU outputs saved at ",os.path.basename(HRU_output_folder) + '.zip')

At this point, we have our final discretization saved as geospatial files (also all zipped up for you). You probably want to save this for later reference!  Simply manage files through Folder icon to the far left - look for land_lake_HRUs.zip file under your network name folder (you named this in Section 2.0 somewhere).

# **8.0 Produce Raven-required inputs**

The above sections gave us a reasonably discretized HRU map. We need to dump the information archived in those shapefiles to ASCII-based files for Raven hydrologic modelling. This produces three Raven input files:
 * 'model_name'.rvh
 * Lakes.rvh
 * channel_properties.rvp
    --> please remember to add:
    ":RedirectToFile channel_properties.rvp"
    in your main Raven rvp file which BasinMaker does not produce.

The BasinMaker function `Generate_Raven_Model_Inputs` will be used.

The parameters and outputs of function `Generate_Raven_Model_Inputs` are defined [here](https://basinmaker.readthedocs.io/en/latest/basinmaker_tools.html#generate-raven-input-files).

After producing the discretization related Raven inputs with this Step there are still Raven files users must create before running the model.  
To run a Raven model, four additional files need to be generated by user, they are:
* .rvi file. This file defines the primary functioning of the Raven model, several templates for different hydrological models can be viewed in the Appendix of the Raven User's Manual or generated using RavenR.

* .rvp file. This is the Raven model parameter file, which can be generated as a template with the `:CreateRVPTemplate` command in Raven.

* .rvt file. This is the Raven model forcing input files.

* .rvc file. A blank file for initial conditions.

A separate tutorial guides folks how to get those other four files created for Raven.

In [ ]:
# One ***USER INPUT***  denoted below.  Look below before pressing play.

# NOTE: the code cell below MUST BE executed after the Generate_HRUs function code cell above is executed successfully.

# define another folder that will save the outputs
raven_model_dir = os.path.join(main_output_directory,watershed_name,network_name,'Raven_inputs')

bm = basinmaker.postprocess()

bm.Generate_Raven_Model_Inputs(
    path_hru_polygon         = os.path.join(HRU_output_folder, "finalcat_hru_info.shp"),
    model_name            ="Mackenzie",  # <-- ***USER INPUT***.  This is used for naming the output files, which are Raven model input files.
    subbasingroup_names_channel   =["Allsubbasins"], # A subbasin group will be created in the rvh file for simultaneous manipulation in Raven modeling.
    subbasingroup_length_channel   =[-1],
    subbasingroup_name_lake      =["AllLakesubbasins"],
    subbasingroup_area_lake      =[-1],
    path_output_folder         = raven_model_dir,
    aspect_from_gis          = 'purepy',
)

# zip file
zipfile = os.path.join(os.path.dirname(raven_model_dir),os.path.basename(raven_model_dir) + '.zip')
!zip -q -r  "$zipfile" "$raven_model_dir"
print("The zipped Raven input files saved at ",os.path.basename(raven_model_dir) + '.zip')

# **Summary of BasinMaker outputs (if you did all Sections above):**

**Warning**: all your BasinMaker outputs are on the Colab temporary drive and will be deleted once you close this webpage. Move desired outputs to your Google Drive (if mounted) or download zipped files of interest.  Assuming network_name=02LE024, your folder '/'content/02LE024/ has subfolders of BasinMaker postprocessing outputs for each of the last 5 steps above:

Step 3 outputs: Extraction or Extraction.zip
- this is the highest resolution lake-river routing network available in CLRH.  You save this if you want to be able to upload later into Colab to simplify/post-process as desired.

Step 5 outputs: filter_lakes or filter_lakes.zip
- this is simplified network after small lakes removed. You save this if you want to be able to upload later into Colab to simplify/post-process as desired.  
- includes network as two required geojsons for viewing output network in RavenView

Step 6 outputs: increase_drainage_area or increase_drainage_areas.zip
- this is simplified network after smaller subbasins combined to form larger subbasins. You save this if you want to be able to upload later into Colab to simplify/post-process as desired.
- this typically describes the final routing network of interest and is thus a folder you likely wish to copy.
- includes CanHydroFabric_Watersheds.shp file of network boundary which can be used to download forecast or historical meteorological data from CaSPAr archive.
- includes network as two required geojsons for viewing output network in RavenView  

Step 7 outputs: land_lake_HRUs or land_lake_HRUs.zip
- this contains the HRU shapefiles that would be needed in semi-distributed hydrological models based on the HRU concept. Step 5 output network is typically the input network.
- forms the inputs required for the write Raven hydrological model input files (Step 7)
- includes an outline.shp file of network boundary which can be used to download forecast or historical meteorological data from CaSPAr archive.
- HRU shapefiles also required to map gridded NetCDF precipitation to HRUs via the Raven Gridweights generator code.
- includes HRUs as geojsons required for viewing HRU level Raven model output in RavenView

Step 8 outputs: Raven_inputs or Raven_inputs.zip
- this contains the Raven hydrological model framework input files BasinMaker can create (lake, channel and HRU properties and geospatial attributes and how they all connect)
- includes routing network as two required geojsons for viewing Raven model output across the routing network in RavenView. In other words, the Raven model input files will generate Raven outputs that are 100% compatible with the RavenView geojsons.   

**Now, download your Raven model inputs!**  Manage download through Folder icon to the far left: Raven_inputs.zip.

# **9.0 APPENDIX OF USEFUL COLAB EXAMPLE CODES**

## **Mount/Unmount your Google Drive on Colab Runtime**

Mount Google Drive

In [ ]:
# This is optional, but recommended for easier data transfer.
# Remember to refersh the file list after mounting.
from google.colab import drive
drive.mount('/content/google_drive')      # The folder name 'google_drive' can be changed by your own preference.

Unmount Google Drive

In [ ]:
# This is optional
# Do NOT run this block unless you want to unmount.
from google.colab import drive
drive.flush_and_unmount()

## **Zip/Unzip files/folders**

Unzip files

In [ ]:
# This is optional
# This code may be used if you directly drag your zipped data file to Colab.
# We will need this unzip code later in the HRU section.
!unzip "/content/google_drive/MyDrive/Colab Notebooks/NCO_Exercise.zip" -d "/content/google_drive/MyDrive/Colab Notebooks"    # XXX.zip is the zip file you want to extract to Colab

Zip files


In [ ]:
# This is optional
# If you want to download files to local drive, you need to zip them first and this code may be applied
# In these exercises below, a zip code block has been added in after each key processing.
# You don't have to run this block unless you want a try.
!zip "your_target_path_for_zipped_file/YYY.zip" "your_target_folder_path_to-be-zipped"      # YYY.zip is the name for the zipped file

## **Remove a folder**

In [ ]:
# This is optional
# If you want to remove a folder during your processing, excute the following code.
# This is useful when you wrongly create a folder and don't want to see it there.
# Adjust the quoted path to the directory you want to remove.
# %rm -rf "/content/02LE024/increase_drainage_area"

# OR:
shutil.rmtree('05CB006')  # all subdirectories too

In [ ]:
import geopandas as gpd

# Read the shapefile and append it to the list
gdf = gpd.read_file(r'D:\Zelalem\CLRH_Basin\CLRH_Basin\01_02\01_02\finalcat_info_riv_v1-0.shp')

# STEP 1: Build the reciprocal set of pairs
pairs = set(zip(gdf['SubId'], gdf['DowSubId']))

# STEP 2: Find rows where DowSubId/SubId are reversed
mask = gdf.apply(lambda r: (r['DowSubId'], r['SubId']) in pairs, axis=1)

# See which rows will be fixed
print("Rows to fix:\n", gdf[mask])

# STEP 3: Fix reciprocal relationships by removing downstream connection
gdf.loc[mask, 'DowSubId'] = -1



In [ ]:
# Create a helper column for easier matching
pairs = set(zip(gdf['SubId'], gdf['DowSubId']))

# Find reciprocal relationships
reciprocal_rows = gdf[
    gdf.apply(lambda row: (row['DowSubId'], row['SubId']) in pairs, axis=1)
]

In [ ]:
# Read the shapefile and append it to the list
gdf = gpd.read_file(r'D:\Zelalem\CLRH_Basin\CLRH_Basin\01_02\01_02\finalcat_info_riv_v1-0.shp')
gdf[gdf['SubId'] == 14002074.0]

In [ ]:
# Remove duplicate rows
gdf1 = gdf.drop_duplicates(subset="SubId", keep="first")

In [ ]:
import numpy as np
np.nanmax(gdf1['SubId'])

In [ ]:
##3 Garbage

In [ ]:
# Function to merge two or more shapefiles into one
import os
import geopandas as gpd
import pandas as pd
#
def merge_shapefiles(
    base_path,
    shapefile_name,
    output_path,
    output_format="gpkg",
    target_crs="EPSG:4326"
):
    """
    Merge multiple shapefiles found across subfolders into one file.

    Parameters
    ----------
    base_path : str
        Directory containing subfolders.
    shapefile_name : str
        Shapefile name you're searching for.
    output_path : str
        Full path for the merged output file.
    output_format : str
        "shp" or "gpkg".
    target_crs : str
        CRS all files will be projected into (e.g., 'EPSG:4326').
    """

    # Validate output format
    output_format = output_format.lower()
    if output_format not in ["shp", "gpkg"]:
        raise ValueError("output_format must be 'shp' or 'gpkg'")

    driver = "ESRI Shapefile" if output_format == "shp" else "GPKG"

    merged_gdf = None

    # List all subfolders
    subfolders = [
        f for f in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, f))
    ]

    for folder in subfolders:
        shp_path = os.path.join(base_path, folder, "increase_drainage_area", shapefile_name)

        if not os.path.exists(shp_path):
            print(f"Missing: {shp_path}")
            continue

        print(f"Reading: {shp_path}")
        gdf = gpd.read_file(shp_path)

        # ----- CRS HANDLING -----
        if gdf.crs is None:
            print(f"⚠ WARNING: {shp_path} has no CRS. Assigning target CRS {target_crs}.")
            gdf = gdf.set_crs(target_crs)
        else:
            # Reproject if different CRS
            if gdf.crs.to_string() != target_crs:
                print(f"Reprojecting {shp_path} from {gdf.crs} to {target_crs}")
                gdf = gdf.to_crs(target_crs)

        # ----- FIX INVALID GEOMETRIES -----
        gdf["geometry"] = gdf["geometry"].buffer(0)

        # ----- MERGE -----
        if merged_gdf is None:
            merged_gdf = gdf
        else:
            merged_gdf = pd.concat([merged_gdf, gdf], ignore_index=True)


    # ----- SAVE RESULT -----
    if merged_gdf is None or merged_gdf.empty:
        print(f"❌ No shapefiles found for: {shapefile_name}")
        return

    merged_gdf.to_file(output_path, driver=driver)
    print(f"✔ Saved merged file: {output_path}")
    print(f"   CRS: {target_crs}")
    print(f"   Format: {output_format.upper()}")


# -------------------------------------------------------------------
# MAIN EXECUTION
# -------------------------------------------------------------------
input_directory = r"D:\Zelalem\CLRH_Basin\BasinMaker"
shapefile_names = [
    "catchment_without_merging_lakes_v1-0.shp",
    "finalcat_info_riv_v1-0.shp",
    "finalcat_info_v1-0.shp",
    "outline.shp",
    "poi_v1-0.shp",
    "river_without_merging_lakes_v1-0.shp",
    "sl_connected_lake_v1-0.shp"
]

# ---- SET OUTPUT FORMAT HERE ----
output_format = "shp"     # "gpkg" or "shp"
# ---- SET TARGET CRS HERE ----
target_crs = "EPSG:4326"   # or your project CRS   # EPSG:3979  # EPSG:4326

for shapefile_name in shapefile_names:
    # output filename automatically adjusted for chosen format
    output_filename = f"merged_{shapefile_name.replace('.shp', f'.{output_format}')}"
    output_path = os.path.join(input_directory, output_filename)
    # merge the same name shapefile
    merge_shapefiles(
        input_directory,
        shapefile_name,
        output_path,
        output_format=output_format,
        target_crs=target_crs
    )